<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week4_online_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 online session – finding a good deal among hotels

Companion notebook to the Week 4 online recording. Run each cell as you watch: every number on the slides is computed here, so you can see the Python behind it.

Data: Békés & Kézdi's Ch. 7 case study – Vienna hotels, November 2017 weekday, 3–4 star hotels (N = 207).

Each section header matches a slide title in the recording.

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_vienna.csv"
hotels = pd.read_csv(data_path)
print(hotels.shape)
hotels[['hotel_id', 'price', 'distance', 'stars', 'rating', 'accommodation_type']].head()

**Columns we use:** `price` = price per night in euros; `distance` = miles from the city centre; `stars` = hotel stars; `rating` = average guest rating (1–5); `accommodation_type` = hotel, apartment, etc.

## The data: one night in Vienna

Keep the textbook's sample: hotels only, actually in Vienna, 3 to 4 stars, no price above €600.

In [ ]:
hotels = hotels[(hotels['accommodation_type'] == 'Hotel') &
                (hotels['city_actual'] == 'Vienna') &
                (hotels['stars'] >= 3) & (hotels['stars'] <= 4) &
                (hotels['price'] <= 600)].copy()
len(hotels)

## First, each variable on its own

In [ ]:
hotels[['price', 'distance']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(hotels['price'], bins=25, color='#457b9d', edgecolor='white')
axes[0].set(title='Price', xlabel='Price per night (€)', ylabel='Hotels')
axes[1].hist(hotels['distance'], bins=20, color='#2d6a4f', edgecolor='white')
axes[1].set(title='Distance', xlabel='Miles from centre', ylabel='Hotels')
plt.tight_layout(); plt.show()

## Bin-scatter: the conditional mean, coarse to finer

In [ ]:
# 2 bins
hotels['bin2'] = np.where(hotels['distance'] < 2, '0-2 miles', '2+ miles')
hotels.groupby('bin2')['price'].agg(['mean', 'count']).round(1)

In [ ]:
# 4 bins, cut-points we choose (right=False: each bin includes its lower edge)
hotels['bin4'] = pd.cut(hotels['distance'], [0, 1, 2, 3, 7], right=False,
                        labels=['0-1', '1-2', '2-3', '3+'])
bin4 = hotels.groupby('bin4', observed=True)['price'].agg(['mean', 'count'])
bin4.round(1)

In [ ]:
plt.figure(figsize=(7, 3.8))
plt.plot(bin4.index.astype(str), bin4['mean'], 'o-', color='#e63946', markersize=9)
plt.xlabel('Distance band (miles)'); plt.ylabel('Mean price (€)'); plt.ylim(0, 150)
plt.tight_layout(); plt.show()

## LOWESS: a smooth version

`frac` is the share of all hotels used in each local average – the bandwidth. Try changing it.

In [ ]:
smooth = lowess(hotels['price'], hotels['distance'], frac=0.3)   # array of (x, fitted y) pairs

plt.figure(figsize=(9, 4))
plt.scatter(hotels['distance'], hotels['price'], s=12, alpha=0.45, color='#457b9d')
plt.plot(smooth[:, 0], smooth[:, 1], color='#e63946', linewidth=3)
plt.xlabel('Distance from centre (miles)'); plt.ylabel('Price per night (€)')
plt.tight_layout(); plt.show()

## The linear regression

In [ ]:
model = smf.ols('price ~ distance', data=hotels).fit()
model.params

In [ ]:
a, b = model.params['Intercept'], model.params['distance']
xs = np.array([0, hotels['distance'].max()])

plt.figure(figsize=(9, 4))
plt.scatter(hotels['distance'], hotels['price'], s=12, alpha=0.45, color='#457b9d')
plt.plot(xs, a + b * xs, color='#1a1a2e', linewidth=3, label='Linear regression')
plt.plot(smooth[:, 0], smooth[:, 1], color='#e63946', linewidth=3, label='LOWESS')
plt.xlabel('Distance from centre (miles)'); plt.ylabel('Price per night (€)'); plt.legend()
plt.tight_layout(); plt.show()

## Residuals

Residual = actual price − predicted price.

In [ ]:
hotels['predicted'] = model.fittedvalues
hotels['residual'] = model.resid      # = price - predicted

print(f"Mean residual: {hotels['residual'].mean():.6f}")   # zero, by construction
plt.figure(figsize=(9, 3.5))
plt.hist(hotels['residual'], bins=30, color='#457b9d', edgecolor='white')
plt.axvline(0, color='#e63946', linestyle='--')
plt.xlabel('Residual (€)'); plt.ylabel('Hotels'); plt.tight_layout(); plt.show()

## The five best deals

In [ ]:
hotels.nsmallest(5, 'residual')[['hotel_id', 'distance', 'stars', 'rating',
                                  'price', 'predicted', 'residual']].round(2)

## How good is the fit?

In [ ]:
print(f"R-squared: {model.rsquared:.3f}")

# Same thing from Week 4's definition: Var[y-hat] / Var[y]
print(f"Var[predicted] / Var[price]: {hotels['predicted'].var() / hotels['price'].var():.3f}")

## Interpolation vs. extrapolation

In [ ]:
print(f"Furthest hotel in the sample: {hotels['distance'].max()} miles")
print(f"Predicted price at 10 miles: €{a + b * 10:.1f}")